# 34. Embeddings, semantic similarity, and named entities

**Level:** Intermediate

**Prerequisites:** 07 and 32-33: vectors and text features.

[Course index](00_START_HERE.ipynb) | [Previous](33_Bag_of_Words_TFIDF_and_Sentiment.ipynb) | [Next](35_Attention_and_Causal_Masking.ipynb)

**Topics:** Embeddings, Vector representations, Semantic similarity, Cosine similarity, Named entity recognition.

**Matching script:** `examples\34_Embeddings_Similarity_and_Named_Entities.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

An embedding maps an item to a numeric vector. Learned embeddings can place items with similar
training contexts near each other, but meaning depends on the training objective and domain.
A manually assigned vector is a demonstration, not a pretrained semantic model.

Cosine similarity compares vector direction after normalizing magnitudes. It does not prove
factual entailment, identity, or correctness. Zero vectors require explicit handling.
Sparse TF-IDF and dense learned embeddings are different representations; both can support
retrieval with different tradeoffs.

Named entity recognition identifies spans such as people, organizations, locations, or domain
entities. A model may use sequence tagging with BIO-style labels, contextual embeddings, or
generation. Pattern matching can extract a constrained hostname/IP format but should not be
marketed as learned general NER. This lesson contrasts those mechanisms explicitly.

## Mathematics or data contract

cosine(a,b)=dot(a,b)/(norm(a)*norm(b)); a zero norm makes this expression undefined.

## Important limitation

The vectors are handcrafted and the entity extractor is regex-based, not a trained NER model.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Vector similarity and constrained entity spans

**Purpose:** Inspect geometry and extraction without hidden downloads.

### Read the code in this order
1. Normalize small handcrafted vectors.
2. Rank a query by cosine similarity.
3. Extract only the declared hostname pattern.
4. Check exact matched spans rather than inferring general language understanding.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
handcrafted similarities: [1.0, 0.995, 0.0]
pattern-based entities: [('edge-01', 0, 7), ('core-02', 27, 34)]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import re
import numpy as np

vectors = np.array([[1., 0.], [0.95, 0.1], [0., 1.]])
names = ["router", "switch", "flower"]
normalized = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
similarities = normalized @ normalized[0]
ranking = np.argsort(-similarities)
assert [names[index] for index in ranking] == ["router", "switch", "flower"]
text = "edge-01 reported loss near core-02."
entities = [(match.group(), match.start(), match.end())
            for match in re.finditer(r"\b(?:edge|core)-\d{2}\b", text)]
assert [item[0] for item in entities] == ["edge-01", "core-02"]
print("handcrafted similarities:", np.round(similarities, 3).tolist())
print("pattern-based entities:", entities)

handcrafted similarities: [1.0, 0.995, 0.0]
pattern-based entities: [('edge-01', 0, 7), ('core-02', 27, 34)]


### Interpret example 1

The result should demonstrate this purpose: **Inspect geometry and extraction without hidden downloads.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Learn lexical latent embeddings from a local corpus

**Purpose:** Distinguish fitted latent semantic representations from handcrafted vectors and pretrained contextual embeddings.

### Read the code in this order
1. Fit TF-IDF on a tiny local corpus.
2. Learn a low-rank SVD representation.
3. Normalize document embeddings and inspect similarity.
4. Verify shape and finite values without claiming broad semantic understanding.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
learned lexical embedding shape: (6, 2)
router-switch / router-garden similarity: 1.0 -0.0
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

corpus = ["router packet forwarding network", "switch packet network port",
          "router switch network", "garden flower plant soil",
          "flower garden leaf plant", "soil garden growing plant"]
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)
model = TruncatedSVD(n_components=2, random_state=42)
embeddings = normalize(model.fit_transform(X))
similarities = embeddings @ embeddings.T
assert embeddings.shape == (6, 2) and np.isfinite(embeddings).all()
assert similarities[0, 1] > similarities[0, 3]
print("learned lexical embedding shape:", embeddings.shape)
print("router-switch / router-garden similarity:", round(similarities[0, 1], 3), round(similarities[0, 3], 3))

learned lexical embedding shape: (6, 2)
router-switch / router-garden similarity: 1.0 -0.0


### Interpret example 2

The result should demonstrate this purpose: **Distinguish fitted latent semantic representations from handcrafted vectors and pretrained contextual embeddings.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 3: Train a constrained token-level entity classifier

**Purpose:** Practice supervised entity labels without pretending a regex is a learned NER model.

### Read the code in this order
1. Represent tokens by explicit lexical shape features.
2. Train a token classifier on synthetic labels.
3. Evaluate previously unseen host IDs and ordinary words.
4. State that there is no general NER or span-context capability here.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `predict(...)` returns the estimator's chosen output, not a guarantee of correctness.
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
synthetic token entity labels: [('edge-99', 'HOST'), ('reported', 'O'), ('core-88', 'HOST')]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def features(token):
    return {"has_dash": "-" in token, "has_digit": any(char.isdigit() for char in token),
            "prefix": token.split("-")[0] if "-" in token else "ordinary"}
train_tokens = ["edge-01", "edge-02", "core-01", "core-02", "lab-01", "lab-02",
                "router", "status", "up", "down", "traffic", "failed"]
targets = ["HOST"] * 6 + ["O"] * 6
model = make_pipeline(DictVectorizer(), LogisticRegression(C=10))
model.fit([features(token) for token in train_tokens], targets)
tokens = ["edge-99", "reported", "core-88"]
predicted = model.predict([features(token) for token in tokens]).tolist()
assert predicted == ["HOST", "O", "HOST"]
print("synthetic token entity labels:", list(zip(tokens, predicted)))

synthetic token entity labels: [('edge-99', 'HOST'), ('reported', 'O'), ('core-88', 'HOST')]


### Interpret example 3

The result should demonstrate this purpose: **Practice supervised entity labels without pretending a regex is a learned NER model.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Evaluate an entity extractor against labeled spans and count exact-span precision and recall.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Embeddings` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: cosine(a,b)=dot(a,b)/(norm(a)*norm(b)); a zero norm makes this expression undefined.
- Name the boundary of the demonstration: The vectors are handcrafted and the entity extractor is regex-based, not a trained NER model.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.